# 第2–7章网页适配验证
教材 pp.18–100。验证全部18个工作区及模型分支；保持原始DataFrame不变。固定种子样本不是实际行情。

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'service').is_dir(): root=root.parent
sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from controller.chapter_groups.risk import operations, examples, example_frame, calculate

def run(operation, parameters=None, frame=None):
    sample, metadata=example_frame(operation)
    if frame is not None: sample=frame
    before=sample.copy(deep=True)
    result=calculate(sample,metadata,{'operation':operation,'parameters':parameters or {}})
    pd.testing.assert_frame_equal(sample,before)
    return result

def metrics(result): return {m['name']:m['value'] for m in result['metrics']}
for op in operations():
    result=run(op['id'])
    assert result['metrics'] or result['tables']
    assert op['chapter'] in range(2,8)
print('全部18个默认操作通过')


全部18个默认操作通过


In [2]:
# 第2章：所有分布/情景/重采样路径；历史尾部边界只取部分概率质量。
for model in ['historical','age','weighted','kernel']:
    assert metrics(run('ch2_distribution',{'model':model}))['ES'] >= metrics(run('ch2_distribution',{'model':model}))['VaR']
for method in ['levels','volatility','correlation','filtered']:
    run('ch2_scenarios',{'method':method})
for estimator in ['var','es','mean']:
    first=run('ch2_bootstrap',{'estimator':estimator,'block_size':3,'n_resamples':20})
    assert first==run('ch2_bootstrap',{'estimator':estimator,'block_size':3,'n_resamples':20})
small=pd.DataFrame({'loss':[1.,2.,3.,4.],'weight':[1.,1.,1.,1.]})
assert_allclose(metrics(run('ch2_distribution',{'confidence_level':.625},small))['ES'],11/3)
run('ch2_garch',{'method':'fit','n_paths':10})
print('第2章通过')


第2章通过


In [3]:
# 第3章：教材例3.6及GEV母分布关系、拟合分支。
from scipy.stats import genextreme
sample=pd.DataFrame({'loss':genextreme.ppf(np.arange(1,301)/301,0,loc=2,scale=.7)})
for method in ['specified','mle','moments','regression','quantile']:
    result=run('ch3_gev',{'method':method},sample)
    if method=='regression': assert_allclose([metrics(result)['location'],metrics(result)['scale']],[2,.7])
raw=pd.DataFrame({'loss':np.arange(1000,dtype=float),'block':np.arange(1000)//100})
run('ch3_gev',{'sample_mode':'raw'},raw)
result=metrics(run('ch3_pot'))
assert_allclose([result['VaR'],result['ES']],[3.952,5.238],atol=.001)
for method in ['mle','moments']: run('ch3_pot',{'method':method})
pareto=pd.DataFrame({'loss':1/np.arange(1,101,dtype=float)**.4})
assert_allclose(metrics(run('ch3_tail',frame=pareto))['Pickands'],.4)
print('第3章通过')


第3章通过


In [4]:
# 第4章：异常率恰好等于预测率时Kupiec统计量为0；教材交通灯。
hits=pd.DataFrame({'hit':[0]*95+[1]*5})
r=run('ch4_coverage',{'input_mode':'hits'},hits)
assert_allclose(r['tables'][0]['rows'][0]['statistic'],0,atol=1e-12)
assert metrics(run('ch4_counts'))['add_on']==.65
assert 'zone' not in metrics(run('ch4_counts',{'sample_size':100}))
# 第5/6章：Euler加总、折现平价、FRA手算和有限差分导数。
r=run('ch5_linear')
assert_allclose(sum(v['component_var'] for v in r['tables'][0]['rows']),metrics(r)['VaR'])
assert_allclose(metrics(run('ch5_cashflows'))['总现值'],100)
assert_allclose(metrics(run('ch5_derivatives',{'instrument':'fra','notional':1e6}))['计算值'],4850)
run('ch5_derivatives',{'instrument':'swap'})
rows=run('ch5_delta_gamma')['tables'][0]['rows']
assert_allclose([v['pnl'] for v in rows],[-.4,0,.6])
derivatives=run('ch6_sensitivity')['tables'][0]['rows']
components=r['tables'][0]['rows']
assert_allclose([v['first_derivative'] for v in derivatives],[v['marginal_var'] for v in components],rtol=1e-7)
a=metrics(run('ch6_intervals'))['standard_error']
b=metrics(run('ch6_intervals',{'sample_size':400}))['standard_error']
assert_allclose(a,2*b)
for method in ['order','exceptions']:run('ch6_intervals',{'method':method})
print('第4–6章通过')


第4–6章通过


In [5]:
# 第7章：均匀PIT网格的KS/CVM解析值，各独立性/异常检验分支。
u=pd.DataFrame({'pit':(np.arange(100)+.5)/100})
r=metrics(run('ch7_pit',{'input_mode':'pit','target_column':'pit'},u))
assert_allclose([r['ks'],r['cvm']],[1/200,1/1200])
for method in ['ljung_box','pit_regression','lpm','logit','duration']:
    run('ch7_dependence',{'method':method,'target_column':'pit' if method=='pit_regression' else 'hit'})
# 拒绝无效边界；不会悄悄裁剪/删行。
for operation,parameters,frame in [
    ('ch7_pit',{'input_mode':'pit','target_column':'pit'},pd.DataFrame({'pit':[0.,.2,.5,.9]})),
    ('ch4_coverage',{'input_mode':'hits'},pd.DataFrame({'hit':[0,2]})),
    ('ch3_tail',{'tail_count':100},None),
    ('ch5_linear',{'covariance':[[1,2],[2,1]]},None),
    ('ch2_bootstrap',{'n_resamples':2.5},None),
]:
    try: run(operation,parameters,frame)
    except ValueError: pass
    else: raise AssertionError(operation)
print('第7章及无效边界通过')


第7章及无效边界通过
